In [1]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import random
import numpy as np
import pandas as pd
import torch

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)

In [2]:
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA:", torch.version.cuda)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

PyTorch: 2.5.1+cu121
CUDA available: True
GPU: NVIDIA GeForce RTX 4060 Laptop GPU
CUDA: 12.1


device(type='cuda')

In [3]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

In [4]:
ann = pd.read_csv("../data/processed/mhs_main_experiment_annotations_with_split.csv")

print("Annotation-level shape:", ann.shape)
print(ann["split"].value_counts())

Annotation-level shape: (49433, 53)
split
train         34413
test           7713
validation     7307
Name: count, dtype: int64


In [5]:
ann = ann[ann["is_women_targeted"] == 1].copy()

print("Women-targeted annotation rows:", ann.shape)
print("Unique comments:", ann["comment_id"].nunique())
print(ann["split"].value_counts())

Women-targeted annotation rows: (27889, 53)
Unique comments: 11983
split
train         19124
test           4771
validation     3994
Name: count, dtype: int64


In [6]:
target_col = "hatespeech"
label_values = [0, 1, 2]

def make_distribution(group, label_col=target_col, label_values=label_values):
    counts = group[label_col].value_counts(normalize=True)

    return pd.Series({
        f"{label_col}_{label}_prob": counts.get(label, 0.0)
        for label in label_values
    })

In [7]:
global_dist = (
    ann.groupby("comment_id")
    .apply(lambda g: make_distribution(g))
    .reset_index()
)

In [8]:
women_dist = (
    ann[ann["annotator_gender_group"] == "women"]
    .groupby("comment_id")
    .apply(lambda g: make_distribution(g))
    .reset_index()
    .rename(columns={
        "hatespeech_0_prob": "women_hatespeech_0_prob",
        "hatespeech_1_prob": "women_hatespeech_1_prob",
        "hatespeech_2_prob": "women_hatespeech_2_prob"
    })
)

In [9]:
men_dist = (
    ann[ann["annotator_gender_group"] == "men"]
    .groupby("comment_id")
    .apply(lambda g: make_distribution(g))
    .reset_index()
    .rename(columns={
        "hatespeech_0_prob": "men_hatespeech_0_prob",
        "hatespeech_1_prob": "men_hatespeech_1_prob",
        "hatespeech_2_prob": "men_hatespeech_2_prob"
    })
)

In [10]:
metadata = (
    ann.groupby("comment_id")
    .agg(
        text_clean=("text_clean", "first"),
        split=("split", "first"),
        target_type=("target_type", "first"),
        annotation_count=("annotator_id", "count"),
        unique_annotators=("annotator_id", "nunique")
    )
    .reset_index()
)

In [11]:
comment_df = (
    metadata
    .merge(global_dist, on="comment_id", how="left")
    .merge(women_dist, on="comment_id", how="left")
    .merge(men_dist, on="comment_id", how="left")
)

print(comment_df.shape)
comment_df.head()

(11983, 15)


,comment_id,text_clean,split,target_type,annotation_count,unique_annotators,hatespeech_0_prob,hatespeech_1_prob,hatespeech_2_prob,women_hatespeech_0_prob,women_hatespeech_1_prob,women_hatespeech_2_prob,men_hatespeech_0_prob,men_hatespeech_1_prob,men_hatespeech_2_prob
0,6,First off you look cool as fuck! Anyway if we ...,train,women_only,2,2,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
1,11,"eat my fuck, bitch",validation,women_only,2,2,0.0,0.5,0.5,0.0,1.0,0.0,0.0,0.0,1.0
2,12,She ugly anyways,train,women_only,2,2,0.5,0.0,0.5,0.5,0.0,0.5,NaN,NaN,NaN
3,19,Won't happen. Birth rates are down. Women want...,train,women_only,2,2,1.0,0.0,0.0,NaN,NaN,NaN,1.0,0.0,0.0
4,22,The guys are one thing but then you have a wom...,train,women_only,2,2,0.5,0.0,0.5,NaN,NaN,NaN,0.5,0.0,0.5


In [12]:
global_cols = [
    "hatespeech_0_prob",
    "hatespeech_1_prob",
    "hatespeech_2_prob"
]

women_cols = [
    "women_hatespeech_0_prob",
    "women_hatespeech_1_prob",
    "women_hatespeech_2_prob"
]

men_cols = [
    "men_hatespeech_0_prob",
    "men_hatespeech_1_prob",
    "men_hatespeech_2_prob"
]

comment_df["global_mask"] = comment_df[global_cols].notna().all(axis=1).astype(int)
comment_df["women_mask"] = comment_df[women_cols].notna().all(axis=1).astype(int)
comment_df["men_mask"] = comment_df[men_cols].notna().all(axis=1).astype(int)

comment_df[women_cols + men_cols] = comment_df[women_cols + men_cols].fillna(0.0)

print("Global available:", comment_df["global_mask"].sum())
print("Women available:", comment_df["women_mask"].sum())
print("Men available:", comment_df["men_mask"].sum())

comment_df[["global_mask", "women_mask", "men_mask"]].value_counts()

Global available: 11983
Women available: 8630
Men available: 7168


global_mask  women_mask  men_mask
1            1           0           4761
                         1           3869
             0           1           3299
                         0             54
Name: count, dtype: int64

In [13]:
rows = []

def add_example(comment_id, text, split, perspective, probs):
    rows.append({
        "comment_id": comment_id,
        "text": f"[{perspective}] {text}",
        "split": split,
        "perspective": perspective,
        "hatespeech_0_prob": probs[0],
        "hatespeech_1_prob": probs[1],
        "hatespeech_2_prob": probs[2]
    })

for _, row in comment_df.iterrows():

    # Global perspective always exists
    add_example(
        comment_id=row["comment_id"],
        text=row["text_clean"],
        split=row["split"],
        perspective="GLOBAL",
        probs=[
            row["hatespeech_0_prob"],
            row["hatespeech_1_prob"],
            row["hatespeech_2_prob"]
        ]
    )

    # Women perspective only if available
    if row["women_mask"] == 1:
        add_example(
            comment_id=row["comment_id"],
            text=row["text_clean"],
            split=row["split"],
            perspective="WOMEN",
            probs=[
                row["women_hatespeech_0_prob"],
                row["women_hatespeech_1_prob"],
                row["women_hatespeech_2_prob"]
            ]
        )

    # Men perspective only if available
    if row["men_mask"] == 1:
        add_example(
            comment_id=row["comment_id"],
            text=row["text_clean"],
            split=row["split"],
            perspective="MEN",
            probs=[
                row["men_hatespeech_0_prob"],
                row["men_hatespeech_1_prob"],
                row["men_hatespeech_2_prob"]
            ]
        )

pc_df = pd.DataFrame(rows)

print("Perspective-conditioned rows:", pc_df.shape)
pc_df["perspective"].value_counts()

Perspective-conditioned rows: (27781, 7)


perspective
GLOBAL    11983
WOMEN      8630
MEN        7168
Name: count, dtype: int64

In [14]:
train_df = pc_df[pc_df["split"] == "train"].copy()
val_df = pc_df[pc_df["split"] == "validation"].copy()
test_df = pc_df[pc_df["split"] == "test"].copy()

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

print("\nTrain perspectives:")
print(train_df["perspective"].value_counts())

print("\nValidation perspectives:")
print(val_df["perspective"].value_counts())

print("\nTest perspectives:")
print(test_df["perspective"].value_counts())

Train: (19428, 7)
Validation: (4187, 7)
Test: (4166, 7)

Train perspectives:
perspective
GLOBAL    8387
WOMEN     6021
MEN       5020
Name: count, dtype: int64

Validation perspectives:
perspective
GLOBAL    1797
WOMEN     1329
MEN       1061
Name: count, dtype: int64

Test perspectives:
perspective
GLOBAL    1799
WOMEN     1280
MEN       1087
Name: count, dtype: int64


In [15]:
label_cols = [
    "hatespeech_0_prob",
    "hatespeech_1_prob",
    "hatespeech_2_prob"
]

for name, part in [("train", train_df), ("validation", val_df), ("test", test_df)]:
    sums = part[label_cols].sum(axis=1)
    print(name)
    print("min:", sums.min())
    print("max:", sums.max())
    print("mean:", sums.mean())
    print()

train
min: 0.9999999999999999
max: 1.0
mean: 1.0

validation
min: 1.0
max: 1.0
mean: 1.0

test
min: 1.0
max: 1.0
mean: 1.0



In [16]:
hf_cols = ["text"] + label_cols + ["perspective"]

train_dataset = Dataset.from_pandas(train_df[hf_cols].reset_index(drop=True))
val_dataset = Dataset.from_pandas(val_df[hf_cols].reset_index(drop=True))
test_dataset = Dataset.from_pandas(test_df[hf_cols].reset_index(drop=True))

train_dataset

Dataset({
    features: ['text', 'hatespeech_0_prob', 'hatespeech_1_prob', 'hatespeech_2_prob', 'perspective'],
    num_rows: 19428
})

In [17]:
model_name = "roberta-base"
MAX_LENGTH = 128

tokenizer = AutoTokenizer.from_pretrained(model_name)

special_tokens = {
    "additional_special_tokens": [
        "[GLOBAL]",
        "[WOMEN]",
        "[MEN]"
    ]
}

num_added = tokenizer.add_special_tokens(special_tokens)

print("Added special tokens:", num_added)

Added special tokens: 3


In [18]:
def tokenize(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=MAX_LENGTH
    )

train_dataset = train_dataset.map(tokenize, batched=True)
val_dataset = val_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)

Map:   0%|          | 0/19428 [00:00<?, ? examples/s]

Map:   0%|          | 0/4187 [00:00<?, ? examples/s]

Map:   0%|          | 0/4166 [00:00<?, ? examples/s]

In [19]:
def add_labels(batch):
    labels = []

    for i in range(len(batch[label_cols[0]])):
        labels.append([
            float(batch["hatespeech_0_prob"][i]),
            float(batch["hatespeech_1_prob"][i]),
            float(batch["hatespeech_2_prob"][i])
        ])

    batch["labels"] = labels
    return batch

train_dataset = train_dataset.map(add_labels, batched=True)
val_dataset = val_dataset.map(add_labels, batched=True)
test_dataset = test_dataset.map(add_labels, batched=True)

Map:   0%|          | 0/19428 [00:00<?, ? examples/s]

Map:   0%|          | 0/4187 [00:00<?, ? examples/s]

Map:   0%|          | 0/4166 [00:00<?, ? examples/s]

In [20]:
columns_to_keep = [
    "input_ids",
    "attention_mask",
    "labels"
]

train_dataset.set_format(type="torch", columns=columns_to_keep)
val_dataset.set_format(type="torch", columns=columns_to_keep)
test_dataset.set_format(type="torch", columns=columns_to_keep)

In [21]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [22]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=3
)

model.resize_token_embeddings(len(tokenizer))
model.to(device)

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


RobertaForSequenceClassification(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50268, 768, padding_idx=1)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
         

In [23]:
class SoftLabelTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop("labels").float()

        outputs = model(**inputs)
        logits = outputs.logits

        log_probs = torch.nn.functional.log_softmax(logits, dim=-1)

        loss = torch.nn.functional.kl_div(
            log_probs,
            labels,
            reduction="batchmean"
        )

        return (loss, outputs) if return_outputs else loss

In [24]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred

    logits = torch.tensor(logits)
    labels = np.array(labels)

    probs = torch.nn.functional.softmax(logits, dim=-1).numpy()

    eps = 1e-12

    kl = np.sum(
        labels * (np.log(labels + eps) - np.log(probs + eps)),
        axis=1
    ).mean()

    m = 0.5 * (labels + probs)

    js = 0.5 * (
        np.sum(labels * (np.log(labels + eps) - np.log(m + eps)), axis=1)
        +
        np.sum(probs * (np.log(probs + eps) - np.log(m + eps)), axis=1)
    ).mean()

    pred_hard = probs.argmax(axis=1)
    gold_hard = labels.argmax(axis=1)

    hard_accuracy = (pred_hard == gold_hard).mean()

    pred_entropy = -np.sum(probs * np.log2(probs + eps), axis=1)
    gold_entropy = -np.sum(labels * np.log2(labels + eps), axis=1)

    if np.std(pred_entropy) == 0 or np.std(gold_entropy) == 0:
        entropy_corr = np.nan
    else:
        entropy_corr = np.corrcoef(pred_entropy, gold_entropy)[0, 1]

    return {
        "kl_divergence": float(kl),
        "js_divergence": float(js),
        "hard_accuracy": float(hard_accuracy),
        "entropy_correlation": float(entropy_corr)
    }

In [25]:
training_args = TrainingArguments(
    output_dir="../models/roberta_perspective_conditioned_gender_1",

    eval_strategy="epoch",
    save_strategy="epoch",

    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,

    num_train_epochs=10,
    weight_decay=0.01,

    logging_steps=100,
    load_best_model_at_end=True,
    metric_for_best_model="js_divergence",
    greater_is_better=False,

    fp16=torch.cuda.is_available(),
    report_to="none",
    seed=SEED
)

In [26]:
trainer = SoftLabelTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

/home/shayan/Distributional-Hate-Speech-Prediction/.venv/lib/python3.12/site-packages/accelerate/accelerator.py:494: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)


In [ ]:
torch.cuda.empty_cache()

trainer.train()

Epoch,Training Loss,Validation Loss,Kl Divergence,Js Divergence,Hard Accuracy,Entropy Correlation
1,0.584900,0.643044,0.643044,0.148950,0.722713,0.208532
2,0.426200,0.648494,0.648494,0.155173,0.703845,0.214730
3,0.390000,0.738764,0.738764,0.158478,0.713160,0.183725
4,0.336300,0.767274,0.767274,0.160895,0.704323,0.200528
5,0.255400,1.076662,1.076662,0.169899,0.693337,0.180113


In [ ]:
val_results = trainer.evaluate(val_dataset)
val_results

In [ ]:
test_results = trainer.evaluate(test_dataset)
test_results

In [ ]:
def predict_dataset(trainer, dataset, original_df):
    pred_output = trainer.predict(dataset)

    logits = pred_output.predictions
    labels = pred_output.label_ids

    probs = torch.nn.functional.softmax(
        torch.tensor(logits),
        dim=-1
    ).numpy()

    out = original_df.reset_index(drop=True).copy()

    out["pred_0"] = probs[:, 0]
    out["pred_1"] = probs[:, 1]
    out["pred_2"] = probs[:, 2]

    out["gold_0"] = labels[:, 0]
    out["gold_1"] = labels[:, 1]
    out["gold_2"] = labels[:, 2]

    return out

In [ ]:
test_pred_df = predict_dataset(
    trainer,
    test_dataset,
    test_df
)

test_pred_df.head()

In [ ]:
def metrics_for_frame(frame):
    probs = frame[["pred_0", "pred_1", "pred_2"]].values
    labels = frame[["gold_0", "gold_1", "gold_2"]].values

    eps = 1e-12

    kl = np.sum(
        labels * (np.log(labels + eps) - np.log(probs + eps)),
        axis=1
    ).mean()

    m = 0.5 * (labels + probs)

    js = 0.5 * (
        np.sum(labels * (np.log(labels + eps) - np.log(m + eps)), axis=1)
        +
        np.sum(probs * (np.log(probs + eps) - np.log(m + eps)), axis=1)
    ).mean()

    hard_accuracy = (
        probs.argmax(axis=1) == labels.argmax(axis=1)
    ).mean()

    pred_entropy = -np.sum(probs * np.log2(probs + eps), axis=1)
    gold_entropy = -np.sum(labels * np.log2(labels + eps), axis=1)

    if np.std(pred_entropy) == 0 or np.std(gold_entropy) == 0:
        entropy_corr = np.nan
    else:
        entropy_corr = np.corrcoef(pred_entropy, gold_entropy)[0, 1]

    return pd.Series({
        "n_examples": len(frame),
        "kl_divergence": kl,
        "js_divergence": js,
        "hard_accuracy": hard_accuracy,
        "entropy_correlation": entropy_corr
    })

In [ ]:
perspective_test_results = (
    test_pred_df
    .groupby("perspective")
    .apply(metrics_for_frame)
    .reset_index()
)

perspective_test_results

In [ ]:
model_save_path = "../models/roberta_perspective_conditioned_gender_1/final_model"

trainer.save_model(model_save_path)
tokenizer.save_pretrained(model_save_path)

print("Saved model to:", model_save_path)

In [ ]:
from datetime import datetime

os.makedirs("../results/tables", exist_ok=True)

results_path = "../results/tables/roberta_perspective_conditioned_gender_1_results.txt"
history_path = "../results/tables/roberta_perspective_conditioned_gender_1_training_history.csv"

history_df = pd.DataFrame(trainer.state.log_history)
history_df.to_csv(history_path, index=False)

with open(results_path, "w", encoding="utf-8") as f:
    f.write("ROBERTA PERSPECTIVE-CONDITIONED GENDER MODEL RESULTS\n")
    f.write("=" * 80 + "\n\n")

    f.write("1. RUN INFORMATION\n")
    f.write("-" * 80 + "\n")
    f.write(f"Run timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
    f.write(f"Model name: {model_name}\n")
    f.write(f"Output directory: {training_args.output_dir}\n")
    f.write(f"Best model checkpoint: {trainer.state.best_model_checkpoint}\n")
    f.write(f"Best validation metric: {trainer.state.best_metric}\n")
    f.write(f"Global training steps: {trainer.state.global_step}\n\n")

    f.write("2. MODEL OBJECTIVE\n")
    f.write("-" * 80 + "\n")
    f.write("Dataset: women-targeted comments only\n")
    f.write("Architecture: perspective-conditioned RoBERTa\n")
    f.write("Input format: [PERSPECTIVE] comment_text\n")
    f.write("Perspectives: [GLOBAL], [WOMEN], [MEN]\n")
    f.write("Output: 3-class hatespeech soft-label distribution\n\n")

    f.write("3. DATASET SIZES\n")
    f.write("-" * 80 + "\n")
    f.write(f"Perspective-conditioned total rows: {len(pc_df)}\n")
    f.write(f"Train rows: {len(train_df)}\n")
    f.write(f"Validation rows: {len(val_df)}\n")
    f.write(f"Test rows: {len(test_df)}\n\n")

    f.write("Perspective counts:\n")
    f.write(str(pc_df["perspective"].value_counts()))
    f.write("\n\n")

    f.write("4. TRAINING SETUP\n")
    f.write("-" * 80 + "\n")
    f.write(f"Max sequence length: {MAX_LENGTH}\n")
    f.write(f"Train batch size: {training_args.per_device_train_batch_size}\n")
    f.write(f"Eval batch size: {training_args.per_device_eval_batch_size}\n")
    f.write(f"Epochs: {training_args.num_train_epochs}\n")
    f.write(f"Learning rate: {training_args.learning_rate}\n")
    f.write(f"Weight decay: {training_args.weight_decay}\n")
    f.write(f"FP16: {training_args.fp16}\n\n")

    f.write("5. VALIDATION RESULTS\n")
    f.write("-" * 80 + "\n")
    for key, value in val_results.items():
        f.write(f"{key}: {value}\n")
    f.write("\n")

    f.write("6. TEST RESULTS\n")
    f.write("-" * 80 + "\n")
    for key, value in test_results.items():
        f.write(f"{key}: {value}\n")
    f.write("\n")

    f.write("7. TEST RESULTS BY PERSPECTIVE\n")
    f.write("-" * 80 + "\n")
    f.write(perspective_test_results.to_string(index=False))
    f.write("\n\n")

    f.write("8. TRAINING HISTORY\n")
    f.write("-" * 80 + "\n")
    f.write(f"Training history saved to: {history_path}\n")
    f.write(f"Total log entries: {len(history_df)}\n\n")

    f.write("9. NOTES\n")
    f.write("-" * 80 + "\n")
    f.write("This model conditions the prediction on demographic perspective tokens.\n")
    f.write("Unlike the previous multi-head architecture, it uses one shared output head.\n")
    f.write("The goal is to test whether the same text produces different predicted distributions under different perspective prompts.\n")

print("Saved:", results_path)
print("Saved history:", history_path)